# Défi 3 — Survie à 2 mois avec un réseau de neurones

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/ai_for_medicine/blob/main/ia-medecine/notebooks/iam-s3-defi.ipynb)

*IA pour la médecine — séance 3.*

Même défi qu'aux séances 1 et 2 : prédire si un patient de réanimation de l'étude
**SUPPORT2** est vivant (`alive`) ou décédé (`dead`) **2 mois** après son
admission. Métrique : l'**exactitude** ; prédire toujours `alive` donne environ 0,59.

La méthode du jour : un **réseau de neurones avec arrêt précoce**, comparé
honnêtement, sur le même jeu de validation, à la régression logistique (séance 1)
et à la forêt aléatoire (séance 2).

**Soumission** : le fichier `submission.csv` (colonnes `patient_id,outcome`, une
ligne par patient de `X_test`) se dépose sur la page du défi : https://ml-arena.com/viewchallenge/201

## 1. Charger les données

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, recall_score

URL = "https://raw.githubusercontent.com/racousin/ai_for_medicine/main/ia-medecine/donnees/support2/"
X_train = pd.read_csv(URL + "X_train.csv")
y_train = pd.read_csv(URL + "y_train_survie.csv")
X_test = pd.read_csv(URL + "X_test.csv")

assert (X_train["patient_id"] == y_train["patient_id"]).all()
print(f"Entraînement : {len(X_train)} patients ; test : {len(X_test)} patients")
y_train["outcome"].value_counts(normalize=True)

## 2. Préparer les données

La préparation de la séance 2, écrite une seule fois dans un `ColumnTransformer` :

- variables numériques : valeurs manquantes remplacées par la **médiane**, puis
  **standardisation** (indispensable pour le réseau) ;
- variables catégorielles (texte) : valeur la plus fréquente, puis **encodage
  one-hot**.

Placé dans un pipeline avec le modèle, le préparateur est ajusté sur les seuls
patients d'entraînement à chaque `fit` : pas de fuite d'information.

In [ ]:
X = X_train.drop(columns="patient_id")
y = y_train["outcome"]
categorielles = X.select_dtypes(exclude="number").columns.tolist()
numeriques = X.select_dtypes(include="number").columns.tolist()
print("Catégorielles :", categorielles)

def preparation():
    return ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeriques),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                              OneHotEncoder(handle_unknown="ignore")), categorielles),
    ])

X_fit, X_val, y_fit, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Ajustement : {len(X_fit)} patients ; validation : {len(X_val)} patients")

## 3. Les modèles des séances 1 et 2

Deux références, évaluées sur le **même** jeu de validation. On affiche
l'exactitude (la métrique du défi) et le rappel des décès.

In [ ]:
def evaluer(nom, modele):
    pipe = make_pipeline(preparation(), modele).fit(X_fit, y_fit)
    pred = pipe.predict(X_val)
    print(f"{nom:<32} exactitude {accuracy_score(y_val, pred):.4f}   "
          f"rappel des décès {recall_score(y_val, pred, pos_label='dead'):.3f}")
    return pipe

evaluer("Régression logistique", LogisticRegression(max_iter=5000))
evaluer("Forêt aléatoire", RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                                  random_state=42, n_jobs=-1))

## 4. Le réseau de neurones

**À faire** : évaluez deux réseaux de deux couches cachées `(64, 32)`
(`random_state=42`) :

1. sans arrêt précoce (`max_iter=500`) ;
2. avec arrêt précoce : `early_stopping=True`, `validation_fraction=0.1`,
   `n_iter_no_change=10`.

Comparez-les aux deux références. Que fait l'arrêt précoce ?

In [ ]:
# TODO : evaluer("MLP sans arrêt précoce", MLPClassifier(...))
# TODO : mlp = evaluer("MLP avec arrêt précoce", MLPClassifier(..., early_stopping=True, ...))

*Votre réponse ici.*

## 5. Soumission

Le modèle choisi est réentraîné sur **tous** les patients d'entraînement, puis
prédit ceux de `X_test`.

In [ ]:
# TODO : final = make_pipeline(preparation(), MLPClassifier(... avec arrêt précoce ...))
# TODO : final.fit(X, y)
# TODO : submission = pd.DataFrame({"patient_id": X_test["patient_id"],
#                                   "outcome": final.predict(X_test.drop(columns="patient_id"))})
# TODO : submission.to_csv("submission.csv", index=False)

Dans Colab, la cellule suivante télécharge `submission.csv` ; ailleurs, le fichier
est dans le dossier du notebook. Déposez-le sur https://ml-arena.com/viewchallenge/201.

In [ ]:
try:
    from google.colab import files
except ImportError:
    print("Hors de Colab : submission.csv est dans le dossier courant.")
else:
    files.download("submission.csv")